## Mercedes Homography PipelineUpdated with data image paths and ready to run.

In [1]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# ============================================================
# CONFIG (FOR 3848 x 2168 CAMERA IMAGE ONLY)
# ============================================================
PROJECTOR_IMG_PATH = "../data/Aufgabe_2_projection_circles.png"   # 320x80
CAMERA_IMG_PATH    = "../data/Aufgabe_2_photo_circles.png"        # 3848x2168

# Known pattern layout (4 rows x 10 columns)
N_ROWS = 4
N_COLS = 10
N_EXPECTED = N_ROWS * N_COLS

# Camera ROI for 3848x2168 image
CAM_ROI = (940, 1090, 2350, 1620)   # (x0, y0, x1, y1)

# Thresholds
PROJECTOR_THRESH = 50
CAMERA_THRESH    = 180

# Blob area filters
PROJ_AREA_MIN, PROJ_AREA_MAX = 40, 300
CAM_AREA_MIN, CAM_AREA_MAX   = 2200, 7500   # tuned for 3848x2168

# Homography + matching
RANSAC_THRESH = 3.0
MAX_MATCH_DIST = 60.0   # larger pixel gate for full-res image

# Outputs
CSV_OUT = "matched_points_homography_refined_3848x2168.csv"


# ============================================================
# HELPERS
# ============================================================
def detect_centroids(gray_img, threshold_value, area_min, area_max, roi=None, morph_close=False):
    if roi is not None:
        x0, y0, x1, y1 = roi
        crop = gray_img[y0:y1, x0:x1]
    else:
        x0, y0 = 0, 0
        crop = gray_img

    _, bw = cv2.threshold(crop, threshold_value, 255, cv2.THRESH_BINARY)

    if morph_close:
        kernel = np.ones((3, 3), np.uint8)
        bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=2)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(bw, connectivity=8)

    points = []
    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area_min <= area <= area_max:
            cx = centroids[label, 0] + x0
            cy = centroids[label, 1] + y0
            points.append([cx, cy])

    points = np.array(points, dtype=np.float32)

    bw_full = np.zeros_like(gray_img)
    if roi is not None:
        bw_full[y0:y1, x0:x1] = bw
    else:
        bw_full = bw

    return points, bw_full


def sort_points_grid(points_xy, n_rows=4, n_cols=10):
    pts = np.asarray(points_xy, dtype=np.float32)

    if pts.shape[0] != n_rows * n_cols:
        raise ValueError(f"Expected {n_rows*n_cols} points, got {pts.shape[0]}.")

    pts = pts[np.argsort(pts[:, 1])]
    rows = np.array_split(pts, n_rows)

    ordered_rows = []
    for r in rows:
        r = r[np.argsort(r[:, 0])]
        ordered_rows.append(r)

    return np.vstack(ordered_rows)


def draw_points(image_gray, points_xy, labels=None, color=(0, 255, 0), radius=6):
    out = cv2.cvtColor(image_gray, cv2.COLOR_GRAY2BGR)
    for i, p in enumerate(points_xy):
        x, y = int(round(p[0])), int(round(p[1]))
        cv2.circle(out, (x, y), radius, color, 2, cv2.LINE_AA)
        if labels is not None:
            cv2.putText(out, str(labels[i]), (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return out


def draw_roi_box(image_gray, roi, color=(0, 255, 255)):
    x0, y0, x1, y1 = roi
    out = cv2.cvtColor(image_gray, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(out, (x0, y0), (x1, y1), color, 2)
    return out


def overlay_predicted_and_matched(cam_gray, pred_cam_pts, matches):
    out = cv2.cvtColor(cam_gray, cv2.COLOR_GRAY2BGR)

    for p in pred_cam_pts:
        x, y = int(round(p[0])), int(round(p[1]))
        cv2.circle(out, (x, y), 4, (0, 0, 255), -1, cv2.LINE_AA)

    for m in matches:
        p_pred = (int(round(m["u_c_pred"])), int(round(m["v_c_pred"])))
        p_obs  = (int(round(m["u_c"])), int(round(m["v_c"])))
        cv2.line(out, p_pred, p_obs, (255, 0, 0), 1, cv2.LINE_AA)
        cv2.circle(out, p_obs, 8, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(out, str(m["proj_index"]), (p_obs[0] + 6, p_obs[1] - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)

    return out


def main():
    proj_gray = cv2.imread(PROJECTOR_IMG_PATH, cv2.IMREAD_GRAYSCALE)
    cam_gray  = cv2.imread(CAMERA_IMG_PATH, cv2.IMREAD_GRAYSCALE)

    if proj_gray is None:
        raise FileNotFoundError(f"Projector image not found: {PROJECTOR_IMG_PATH}")
    if cam_gray is None:
        raise FileNotFoundError(f"Camera image not found: {CAMERA_IMG_PATH}")

    print("Projector image shape (H,W):", proj_gray.shape)
    print("Camera image shape (H,W):   ", cam_gray.shape)

    if cam_gray.shape[:2] != (2168, 3848):
        raise RuntimeError(f"This code is configured for 3848x2168 camera image, but loaded {cam_gray.shape[1]}x{cam_gray.shape[0]}.")

    proj_pts, proj_bw = detect_centroids(proj_gray, PROJECTOR_THRESH, PROJ_AREA_MIN, PROJ_AREA_MAX, None, True)
    cam_pts, cam_bw = detect_centroids(cam_gray, CAMERA_THRESH, CAM_AREA_MIN, CAM_AREA_MAX, CAM_ROI, False)

    print("Detected projector centroids:", len(proj_pts))
    print("Detected camera centroids:   ", len(cam_pts))

    if len(proj_pts) != N_EXPECTED or len(cam_pts) != N_EXPECTED:
        print("[WARN] Detection count is not 40.")
        print("       Tune CAMERA_THRESH / CAM_ROI / CAM_AREA_MIN/MAX.")
        raise RuntimeError("Grid ordering expects exactly 40 detections in both images.")

    proj_pts_ord = sort_points_grid(proj_pts, N_ROWS, N_COLS)
    cam_pts_ord  = sort_points_grid(cam_pts,  N_ROWS, N_COLS)

    idx_tl = 0
    idx_tr = N_COLS - 1
    idx_bl = (N_ROWS - 1) * N_COLS
    idx_br = N_ROWS * N_COLS - 1
    seed_idx = [idx_tl, idx_tr, idx_bl, idx_br]

    src = proj_pts_ord[seed_idx].astype(np.float32)
    dst = cam_pts_ord[seed_idx].astype(np.float32)

    print("\nSeed correspondences (src -> dst):")
    for k, idx in enumerate(seed_idx):
        print(f"  idx {idx:2d}: P({src[k,0]:.2f}, {src[k,1]:.2f}) -> C({dst[k,0]:.2f}, {dst[k,1]:.2f})")

    H, inlier_mask = cv2.findHomography(src, dst, cv2.RANSAC, RANSAC_THRESH)
    if H is None:
        raise RuntimeError("Homography estimation failed.")

    print("\nHomography H (camera <- projector):")
    print(H)

    proj_cv = proj_pts_ord.reshape(-1, 1, 2).astype(np.float32)
    pred_cam_pts = cv2.perspectiveTransform(proj_cv, H).reshape(-1, 2)

    tree = cKDTree(cam_pts)
    dists, idxs = tree.query(pred_cam_pts, k=1)

    order = np.argsort(dists)
    used_cam = set()
    matches = []

    for i in order:
        d = float(dists[i])
        j = int(idxs[i])

        if d >= MAX_MATCH_DIST:
            continue
        if j in used_cam:
            continue

        used_cam.add(j)

        matches.append({
            "proj_index": int(i),
            "row": int(i // N_COLS),
            "col": int(i % N_COLS),
            "u_p": float(proj_pts_ord[i, 0]),
            "v_p": float(proj_pts_ord[i, 1]),
            "u_c_pred": float(pred_cam_pts[i, 0]),
            "v_c_pred": float(pred_cam_pts[i, 1]),
            "u_c": float(cam_pts[j, 0]),
            "v_c": float(cam_pts[j, 1]),
            "dx_px": float(cam_pts[j, 0] - pred_cam_pts[i, 0]),
            "dy_px": float(cam_pts[j, 1] - pred_cam_pts[i, 1]),
            "reproj_err_px": d,
            "cam_det_index": int(j)
        })

    matches = sorted(matches, key=lambda m: m["proj_index"])

    print(f"\nValid unique matches: {len(matches)} / {N_EXPECTED}")
    if len(matches) == 0:
        raise RuntimeError("No valid matches found.")

    df = pd.DataFrame(matches)
    df.to_csv(CSV_OUT, index=False)
    print(f"Saved CSV: {CSV_OUT}")

    errs = df["reproj_err_px"].values
    print("\nReprojection error stats [px]:")
    print(f"  min    = {np.min(errs):.3f}")
    print(f"  mean   = {np.mean(errs):.3f}")
    print(f"  median = {np.median(errs):.3f}")
    print(f"  max    = {np.max(errs):.3f}")
    print(f"  mean|dx| = {np.mean(np.abs(df['dx_px'].values)):.3f}")
    print(f"  mean|dy| = {np.mean(np.abs(df['dy_px'].values)):.3f}")

    cam_roi_vis = draw_roi_box(cam_gray, CAM_ROI)
    cv2.imwrite("00_camera_with_roi.png", cam_roi_vis)
    cv2.imwrite("01_projector_binary.png", proj_bw)
    cv2.imwrite("02_camera_binary_roi.png", cam_bw)

    proj_vis = draw_points(proj_gray, proj_pts_ord, labels=[str(i) for i in range(len(proj_pts_ord))], color=(0, 255, 0), radius=4)
    cv2.imwrite("03_projector_indexed.png", proj_vis)

    cam_matched_pts = np.array([[m["u_c"], m["v_c"]] for m in matches], dtype=np.float32)
    cam_labels = [str(m["proj_index"]) for m in matches]
    cam_vis = draw_points(cam_gray, cam_matched_pts, labels=cam_labels, color=(0, 255, 255), radius=8)
    cv2.imwrite("04_camera_indexed.png", cam_vis)

    cam_overlay = overlay_predicted_and_matched(cam_gray, pred_cam_pts, matches)
    cv2.imwrite("05_camera_pred_vs_matched.png", cam_overlay)

    plt.figure(figsize=(8, 4))
    plt.plot(df["proj_index"], df["reproj_err_px"], marker="o")
    plt.xlabel("Projector point index")
    plt.ylabel("Reprojection error [px]")
    plt.title("Homography matching reprojection error")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("06_reprojection_error_plot.png", dpi=180)
    plt.close()

    plt.figure(figsize=(6, 4))
    plt.hist(df["reproj_err_px"], bins=10)
    plt.xlabel("Reprojection error [px]")
    plt.ylabel("Count")
    plt.title("Reprojection error distribution")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("07_reprojection_error_hist.png", dpi=180)
    plt.close()

    print("\nSaved PPT images:")
    print("  00_camera_with_roi.png")
    print("  01_projector_binary.png")
    print("  02_camera_binary_roi.png")
    print("  03_projector_indexed.png")
    print("  04_camera_indexed.png")
    print("  05_camera_pred_vs_matched.png")
    print("  06_reprojection_error_plot.png")
    print("  07_reprojection_error_hist.png")


main()


Projector image shape (H,W): (80, 320)
Camera image shape (H,W):    (2168, 3848)
Detected projector centroids: 40
Detected camera centroids:    40

Seed correspondences (src -> dst):
  idx  0: P(25.00, 8.99) -> C(1047.14, 1247.38)
  idx  9: P(295.00, 8.99) -> C(2126.17, 1238.77)
  idx 30: P(40.00, 71.16) -> C(1111.76, 1488.03)
  idx 39: P(311.03, 71.03) -> C(2191.57, 1476.52)

Homography H (camera <- projector):
[[4.07955968e+00 1.00685303e-01 9.45499292e+02]
 [1.37198239e-02 3.93126519e+00 1.21318927e+03]
 [3.68420561e-05 3.09634640e-05 1.00000000e+00]]

Valid unique matches: 40 / 40


Saved CSV: matched_points_homography_refined_3848x2168.csv

Reprojection error stats [px]:
  min    = 0.000
  mean   = 3.219
  median = 3.233
  max    = 6.145
  mean|dx| = 2.375
  mean|dy| = 1.732



Saved PPT images:
  00_camera_with_roi.png
  01_projector_binary.png
  02_camera_binary_roi.png
  03_projector_indexed.png
  04_camera_indexed.png
  05_camera_pred_vs_matched.png
  06_reprojection_error_plot.png
  07_reprojection_error_hist.png
